In [ ]:
# pyright: reportGeneralTypeIssues=false, reportUnknownMemberType=false, reportUnknownVariableType=false, reportUnknownArgumentType=false
# ruff: noqa
# pylint: skip-file

# NHANES Diabetes Prediction — Logistic Regression (25-Feature Dataset)

Logistic Regression clinical baseline trained on the 25-feature dataset (`processed_data_combined_25features.csv`).

Pipeline:
1. **Imputation** — domain-informed (fill-zero, median, PHQ-9 missing flag)
2. **Feature selection** — VIF iterative removal → backward BIC stepwise (statsmodels)
3. **Final model** — sklearn `LogisticRegression(penalty=None, class_weight='balanced')`

Uses the **same train/val/test split** as the LightGBM notebook for direct comparison.

## 1. Setup and Imports

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import wandb
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────

# W&B
WANDB_PROJECT = "Model exploration for Diabetes Prediction"
ENTITY = "fastegiano-tesis"
STUDY_NAME = "lr_metabolic_history_vif_bic"

# Must match LGBM notebook (model_discovery.ipynb)
RANDOM_STATE = 37
TARGET = "has_diabetes_or_prediabetes"

# VIF threshold (10 = standard; existing notebooks used 5)
VIF_THRESHOLD = 10

# Imputation groups
# Features where NaN means "doesn't do this" → fill with 0
FILL_ZERO_FEATURES = [
    "binge_episodes_month",
    "drinks_per_day",
    "drinking_frequency",
    "vigorous_minutes_per_week",
    "moderate_minutes_per_week",
    "is_current_smoker",
    "ever_smoker",
]

# Features where NaN is structural (eligibility-based) → median + missing flag
STRUCTURAL_MISSING_FEATURES = [
    "phq9_score",
    "told_high_bp",            # NaN = refused/don't know, not "no"
    "told_high_cholesterol",   # NaN = refused/don't know, not "no"
]

## 2. Data Loading and Feature Engineering

In [ ]:
data = pd.read_csv("../../../dataset/processed_data_combined_metabolic_history.csv")
print(f"Raw data shape: {data.shape}")
data.head()

In [ ]:
# Drop non-feature columns (same as LGBM notebook)
data.drop(columns="survey_weight", inplace=True, errors="ignore")

# Drop lab-derived targets — NOT features (require blood draw, not available at inference)
data.drop(columns=["lab_positive", "undiagnosed"], inplace=True, errors="ignore")

# ── Feature Engineering (same as LGBM notebook) ──────────────────────────────
data["waist_to_height_ratio"] = data["BMXWAIST"] / data["BMXHT"]
data["age_bmi_interaction"] = data["RIDAGEYR"] * data["BMXBMI"]


def create_age_bins(
    df: pd.DataFrame,
    age_column: str = "RIDAGEYR",
    age_bins: tuple = (18, 45, 65, 79),
    age_labels: tuple = ("young_adult", "middle_age", "senior"),
    elderly_label: str = "elderly",
    unknown_label: str = "age_unknown",
    elderly_top_coded_age: int = 80,
) -> pd.DataFrame:
    """Bin age into categories and one-hot encode."""
    age = df[age_column].copy()
    age_group = pd.Series(index=df.index, dtype="object")

    missing_mask = age.isna()
    elderly_mask = age >= elderly_top_coded_age

    age_group[elderly_mask] = elderly_label
    valid_mask = ~missing_mask & ~elderly_mask
    age_group[valid_mask] = pd.cut(
        age[valid_mask],
        bins=list(age_bins),
        labels=age_labels,
        right=False,
    )
    age_group[missing_mask] = unknown_label

    dummies = pd.get_dummies(age_group, prefix="age", dtype=int)

    unknown_col = f"age_{unknown_label}"
    if unknown_col in dummies.columns and missing_mask.sum() == 0:
        dummies = dummies.drop(columns=[unknown_col])

    return pd.concat([df, dummies], axis=1)


data = create_age_bins(data)
print(f"Data shape after feature engineering: {data.shape}")
print(f"Columns: {list(data.columns)}")

## 3. Train / Val / Test Split

Identical to the LightGBM notebook — same seed, same ratios (≈80/10/10 stratified).

In [ ]:
X = data.drop(columns=[TARGET, "cycle"])
y = data[TARGET]

print(f"X shape: {X.shape}")
print(f"Prevalence: {y.mean():.3f} ({y.mean():.1%})")

# First split: separate out the untouchable test set (matches LGBM notebook)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.10, stratify=y, random_state=RANDOM_STATE
)

# Second split: carve out validation for threshold tuning
# 0.11 of 90% ≈ 10% of total → gives ~80/10/10 split
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.11, stratify=y_trainval, random_state=RANDOM_STATE
)

print(f"\nTrain: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(f"Prevalence — Train: {y_train.mean():.3f}, Val: {y_val.mean():.3f}, Test: {y_test.mean():.3f}")

## 4. Imputation

Fit on training data only, transform val/test. Three strategies:

| Strategy | Features | Rationale |
|---|---|---|
| Fill 0 | Lifestyle/activity columns | NaN = "doesn't do this" |
| Median + flag | `phq9_score` | Structural: only administered to eligible participants |
| Median/Mode | Remaining numeric/categorical | Low missingness, likely random |

In [ ]:
X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()

# ── A: Fill-zero features (informative missingness = "inactive / none") ──────
for col in FILL_ZERO_FEATURES:
    if col in X_train.columns:
        n_missing_train = X_train[col].isna().sum()
        X_train[col] = X_train[col].fillna(0)
        X_val[col] = X_val[col].fillna(0)
        X_test[col] = X_test[col].fillna(0)
        print(f"{col}: filled {n_missing_train} train NaNs with 0")

print()

# ── B: Structural missing → median + binary flag ──────────────────────────────
for col in STRUCTURAL_MISSING_FEATURES:
    if col in X_train.columns:
        flag_col = f"{col}_missing"
        n_missing_train = X_train[col].isna().sum()

        X_train[flag_col] = X_train[col].isna().astype(int)
        X_val[flag_col] = X_val[col].isna().astype(int)
        X_test[flag_col] = X_test[col].isna().astype(int)

        median_val = X_train[col].median()
        X_train[col] = X_train[col].fillna(median_val)
        X_val[col] = X_val[col].fillna(median_val)
        X_test[col] = X_test[col].fillna(median_val)
        print(f"{col}: filled {n_missing_train} train NaNs with median={median_val:.1f}, added '{flag_col}'")

print()

# ── C: Remaining features with NaN ───────────────────────────────────────────
remaining_nan_cols = X_train.columns[X_train.isna().any()].tolist()
numeric_remaining = X_train[remaining_nan_cols].select_dtypes(include=[np.number]).columns.tolist()
cat_remaining = [c for c in remaining_nan_cols if c not in numeric_remaining]

if numeric_remaining:
    num_imputer = SimpleImputer(strategy="median")
    X_train[numeric_remaining] = num_imputer.fit_transform(X_train[numeric_remaining])
    X_val[numeric_remaining] = num_imputer.transform(X_val[numeric_remaining])
    X_test[numeric_remaining] = num_imputer.transform(X_test[numeric_remaining])
    print(f"Median-imputed numeric: {numeric_remaining}")

if cat_remaining:
    cat_imputer = SimpleImputer(strategy="most_frequent")
    X_train[cat_remaining] = cat_imputer.fit_transform(X_train[cat_remaining])
    X_val[cat_remaining] = cat_imputer.transform(X_val[cat_remaining])
    X_test[cat_remaining] = cat_imputer.transform(X_test[cat_remaining])
    print(f"Mode-imputed categorical: {cat_remaining}")

In [ ]:
# ── Verify no NaNs remain ────────────────────────────────────────────────────
assert X_train.isna().sum().sum() == 0, "Train still has NaNs!"
assert X_val.isna().sum().sum() == 0, "Val still has NaNs!"
assert X_test.isna().sum().sum() == 0, "Test still has NaNs!"
print("✅ All NaNs handled")
print(f"Feature count after imputation: {X_train.shape[1]}")
print(f"Features: {list(X_train.columns)}")

## 5. Feature Scaling

Scale continuous features only — binary 0/1 features are left as-is.  
Fit scaler on train only, transform val and test.

In [ ]:
# Identify binary columns (only values in {0, 1}) — do NOT scale these
binary_cols = [
    col for col in X_train.columns
    if set(X_train[col].dropna().unique()).issubset({0, 1, 0.0, 1.0})
]
scale_cols = [col for col in X_train.columns if col not in binary_cols]

print(f"Binary (unscaled): {binary_cols}")
print(f"\nContinuous (scaled): {scale_cols}")

scaler = StandardScaler()
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_val[scale_cols] = scaler.transform(X_val[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

print(f"\nScaled {len(scale_cols)} features, kept {len(binary_cols)} binary features unscaled")

## 6. VIF Analysis — Multicollinearity Removal

Iteratively remove the feature with the highest VIF until all VIF < threshold.  
VIF is run on the scaled training data (scale-independent VIF).

> **Thesis note:** Features expected to be removed — `BMXWT`/`BMXBMI` (correlated with each other and waist), `BMXHT` (feeds into BMI), `age_bmi_interaction` (correlated with age & BMI), `waist_to_height_ratio` (correlated with waist & height), some age bins. Documenting these removals highlights LR's multicollinearity sensitivity versus tree-based models.

In [ ]:
def calculate_vif(X_df: pd.DataFrame) -> pd.DataFrame:
    """Calculate VIF for all features in a dataframe."""
    vif_data = pd.DataFrame()
    vif_data["feature"] = X_df.columns
    vif_data["VIF"] = [
        variance_inflation_factor(X_df.values, i)
        for i in range(X_df.shape[1])
    ]
    return vif_data.sort_values("VIF", ascending=False).reset_index(drop=True)


def iterative_vif_removal(X_df: pd.DataFrame, threshold: float = VIF_THRESHOLD):
    """Iteratively remove highest VIF feature until all below threshold."""
    dropped = []
    X_current = X_df.copy()

    while True:
        vif = calculate_vif(X_current)
        max_vif = vif["VIF"].max()

        if max_vif <= threshold:
            break

        worst_feature = vif.iloc[0]["feature"]
        print(f"Dropping {worst_feature:<35} (VIF={max_vif:.1f})")
        dropped.append((worst_feature, round(max_vif, 2)))
        X_current = X_current.drop(columns=[worst_feature])

    print(f"\n✅ VIF complete. Dropped {len(dropped)} features. {len(X_current.columns)} remain.")
    return X_current, dropped


X_train_vif, vif_dropped = iterative_vif_removal(X_train)

print("\nFinal VIF scores:")
print(calculate_vif(X_train_vif).to_string(index=False))

In [ ]:
# Apply same column selection to val and test
vif_selected_cols = X_train_vif.columns.tolist()
X_val_vif = X_val[vif_selected_cols]
X_test_vif = X_test[vif_selected_cols]

print(f"VIF-selected features ({len(vif_selected_cols)}):")
for col in vif_selected_cols:
    print(f"  - {col}")

## 7. Backward Stepwise Selection with BIC (statsmodels)

Starting with all VIF-clean features, remove one at a time if BIC improves.  
Produces the most parsimonious model with interpretable p-values and confidence intervals.

In [ ]:
def backward_elimination_bic(X: pd.DataFrame, y: pd.Series) -> list:
    """
    Backward elimination using BIC.
    Starts with all features, removes one at a time if BIC improves.
    """
    features = list(X.columns)
    X_with_const = sm.add_constant(X[features])

    model = sm.Logit(y, X_with_const).fit(disp=0, maxiter=200)
    best_bic = model.bic
    print(f"Starting BIC: {best_bic:.2f} with {len(features)} features")

    improved = True
    while improved and len(features) > 1:
        improved = False
        worst_feature = None
        best_new_bic = best_bic

        for feature in features:
            candidate_features = [f for f in features if f != feature]
            X_candidate = sm.add_constant(X[candidate_features])
            try:
                candidate_model = sm.Logit(y, X_candidate).fit(disp=0, maxiter=200)
                if candidate_model.bic < best_new_bic:
                    best_new_bic = candidate_model.bic
                    worst_feature = feature
            except Exception:
                continue  # Skip if model fails to converge

        if worst_feature is not None:
            features.remove(worst_feature)
            best_bic = best_new_bic
            improved = True
            print(f"Dropped {worst_feature:<35} → BIC: {best_bic:.2f} ({len(features)} features)")

    print(f"\n✅ Stepwise complete. {len(features)} features selected.")
    return features


# Run on training data (post-VIF, pre-scaled)
selected_features = backward_elimination_bic(X_train_vif, y_train)

print("\nSelected features:")
for f in selected_features:
    print(f"  - {f}")

In [ ]:
# ── statsmodels summary: coefficients, p-values, CI, AIC/BIC ─────────────────
# Save / screenshot this output for thesis documentation.
X_final_sm = sm.add_constant(X_train_vif[selected_features])
final_sm_model = sm.Logit(y_train, X_final_sm).fit(disp=0)
print(final_sm_model.summary())

## 8. Final Logistic Regression with sklearn

`penalty=None` matches statsmodels (no regularization).  
Coefficients should be close to the statsmodels output above.

In [ ]:
# Apply feature selection to all splits
X_train_final = X_train_vif[selected_features]
X_val_final = X_val_vif[selected_features]
X_test_final = X_test_vif[selected_features]

lr_model = LogisticRegression(
    penalty=None,             # No regularization — matches statsmodels
    class_weight="balanced",  # Handles class imbalance
    max_iter=1000,
    random_state=RANDOM_STATE,
    solver="lbfgs",
)
lr_model.fit(X_train_final, y_train)

print(f"Converged: {lr_model.n_iter_[0]} iterations")
print(f"Coefficients shape: {lr_model.coef_.shape}")
print("\nsklearn coefficients (should be close to statsmodels):")
for name, coef in zip(selected_features, lr_model.coef_[0]):
    print(f"  {name:<35} {coef:+.4f}")

## 9. Threshold Analysis on Validation Set

In [ ]:
# Predict probabilities on validation set
y_val_proba = lr_model.predict_proba(X_val_final)[:, 1]
val_ap = average_precision_score(y_val, y_val_proba)
print(f"Validation AP: {val_ap:.4f}")

print("\n" + "=" * 70)
print("THRESHOLD ANALYSIS (VALIDATION SET)")
print("=" * 70)
print(f"{'Threshold':<12} {'Recall':<12} {'Precision':<12} {'F1':<12} {'Flagged':<15}")
print("-" * 70)

for thresh in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    y_pred_t = (y_val_proba >= thresh).astype(int)
    rec = recall_score(y_val, y_pred_t, zero_division=0)
    prec = precision_score(y_val, y_pred_t, zero_division=0)
    f1 = f1_score(y_val, y_pred_t, zero_division=0)
    flagged = y_pred_t.sum()
    print(f"{thresh:<12.2f} {rec:<12.1%} {prec:<12.1%} {f1:<12.3f} {flagged} ({flagged/len(y_val):.1%})")

# ── Continuous threshold sweep for 2-panel plot ───────────────────────────────
thresh_range = np.linspace(0.05, 0.6, 100)
recall_at_thresh, precision_at_thresh, f1_at_thresh, flagged_pct = [], [], [], []

for t in thresh_range:
    y_pred_t = (y_val_proba >= t).astype(int)
    r = recall_score(y_val, y_pred_t, zero_division=0)
    p = precision_score(y_val, y_pred_t, zero_division=0)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    recall_at_thresh.append(r)
    precision_at_thresh.append(p)
    f1_at_thresh.append(f1)
    flagged_pct.append(y_pred_t.mean())

# Select threshold targeting ~80% recall (same operating point as LGBM)
target_80_idx = np.argmin(np.abs(np.array(recall_at_thresh) - 0.80))
optimal_thresh = thresh_range[target_80_idx]
print(f"\nSelected threshold for ~80% recall: {optimal_thresh:.3f}")

## 10. Test Set Evaluation

Apply selected threshold to the held-out test set. **This is the final reported result.**

In [ ]:
y_test_proba = lr_model.predict_proba(X_test_final)[:, 1]
y_test_pred = (y_test_proba >= optimal_thresh).astype(int)

test_ap = average_precision_score(y_test, y_test_proba)
test_recall = recall_score(y_test, y_test_pred, zero_division=0)
test_precision = precision_score(y_test, y_test_pred, zero_division=0)
test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
cm = confusion_matrix(y_test, y_test_pred)

print(f"\nTest AP:        {test_ap:.4f}")
print(f"Test Recall:    {test_recall:.3f} ({test_recall:.1%})")
print(f"Test Precision: {test_precision:.3f} ({test_precision:.1%})")
print(f"Test F1:        {test_f1:.3f}")
print(f"\nConfusion Matrix (threshold={optimal_thresh:.3f}):")
print(cm)
print(f"  TN={cm[0,0]}  FP={cm[0,1]}")
print(f"  FN={cm[1,0]}  TP={cm[1,1]}")

## 11. Visualizations

In [ ]:
# ── PR Curve + Threshold sweep (2-panel) ─────────────────────────────────────
precision_arr, recall_arr, thresholds_pr = precision_recall_curve(y_test, y_test_proba)
prevalence = y_test.mean()

val_precision_arr, val_recall_arr, _ = precision_recall_curve(y_val, y_val_proba)
val_pr_auc = float(np.trapz(val_precision_arr, val_recall_arr)) * -1  # trapz on reversed recall

fig_pr, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: PR curve on test set
ax1 = axes[0]
ax1.plot(recall_arr, precision_arr, "b-", linewidth=2, label=f"PR Curve (AP={test_ap:.3f})")
ax1.fill_between(recall_arr, precision_arr, alpha=0.2)

# Mark key operating points
target_recalls = [0.80, 0.75, 0.70, 0.50]
colors_pr = ["red", "orange", "green", "purple"]
for target, color in zip(target_recalls, colors_pr):
    idx = np.where(recall_arr[:-1] >= target)[0]
    if len(idx) > 0:
        i = idx[-1]
        thresh = thresholds_pr[i]
        ax1.scatter(
            recall_arr[i], precision_arr[i], c=color, s=100, zorder=5,
            label=f"Recall={target:.0%} (thresh={thresh:.3f}, prec={precision_arr[i]:.1%})",
        )

ax1.axhline(y=prevalence, color="gray", linestyle="--", label=f"Baseline (prevalence={prevalence:.1%})")
ax1.set_xlabel("Recall (Sensitivity)", fontsize=12)
ax1.set_ylabel("Precision (PPV)", fontsize=12)
ax1.set_title("Precision-Recall Curve — Logistic Regression", fontsize=14)
ax1.legend(loc="upper right", fontsize=9)
ax1.set_xlim([0, 1.02])
ax1.set_ylim([0, 1.02])
ax1.grid(True, alpha=0.3)

# Panel 2: Threshold vs metrics (on validation set)
ax2 = axes[1]
ax2.plot(thresh_range, recall_at_thresh, "b-", linewidth=2, label="Recall")
ax2.plot(thresh_range, precision_at_thresh, "r-", linewidth=2, label="Precision")
ax2.plot(thresh_range, f1_at_thresh, "g--", linewidth=2, label="F1 Score")
ax2.plot(thresh_range, flagged_pct, "k:", linewidth=2, label="% Flagged")
ax2.axvline(x=0.5, color="gray", linestyle="--", alpha=0.7, label="Default (0.5)")
ax2.axvline(
    x=optimal_thresh, color="red", linestyle="--", alpha=0.7,
    label=f"80% Recall (thresh={optimal_thresh:.3f})",
)
ax2.set_xlabel("Decision Threshold", fontsize=12)
ax2.set_ylabel("Score", fontsize=12)
ax2.set_title("Metrics vs Decision Threshold (Validation)", fontsize=14)
ax2.legend(loc="center right", fontsize=9)
ax2.set_xlim([0.05, 0.6])
ax2.set_ylim([0, 1.02])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion Matrix at optimal threshold ─────────────────────────────────────
fig_cm, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_test_pred,
    display_labels=["No Diabetes", "Diabetes"],
    ax=ax,
    cmap="Blues",
)
ax.set_title(
    f"Confusion Matrix at Final Threshold\n"
    f"(threshold={optimal_thresh:.3f}, recall={test_recall:.2%}, precision={test_precision:.2%})"
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Coefficient Plot ──────────────────────────────────────────────────────────
coef_df = pd.DataFrame({
    "feature": selected_features,
    "coefficient": lr_model.coef_[0],
}).sort_values("coefficient", key=abs, ascending=True)

fig_coef, ax = plt.subplots(figsize=(10, max(6, len(selected_features) * 0.4)))
colors_coef = ["#4A90D9" if c > 0 else "#D94A4A" for c in coef_df["coefficient"]]
ax.barh(coef_df["feature"], coef_df["coefficient"], color=colors_coef)
ax.set_xlabel("Coefficient (blue=increases risk, red=decreases risk)")
ax.set_title("Logistic Regression Coefficients")
ax.axvline(x=0, color="gray", linewidth=0.5)
plt.tight_layout()
plt.show()

## 12. W&B Logging

In [ ]:
run = wandb.init(
    project=WANDB_PROJECT,
    entity=ENTITY,
    name=STUDY_NAME,
    job_type="evaluation",
    config={
        "model": "LogisticRegression",
        "penalty": None,
        "class_weight": "balanced",
        "solver": "lbfgs",
        "dataset": "processed_data_combined_25features.csv",
        "n_features_initial": X_train.shape[1],
        "n_features_post_vif": len(vif_selected_cols),
        "n_features_final": len(selected_features),
        "selected_features": selected_features,
        "vif_dropped": [f[0] for f in vif_dropped],
        "vif_threshold": VIF_THRESHOLD,
        "imputation_strategy": "fill_zero + median + phq9_missing_flag",
        "optimal_threshold": float(optimal_thresh),
        "random_state": RANDOM_STATE,
        "train_size": len(y_train),
        "val_size": len(y_val),
        "test_size": len(y_test),
        "train_prevalence": float(y_train.mean()),
    },
)

# ── Metrics ───────────────────────────────────────────────────────────────────
y_test_pred_05 = (y_test_proba >= 0.5).astype(int)

wandb.log({
    "val_ap": val_ap,
    "test_ap": test_ap,
    "test_recall": test_recall,
    "test_precision": test_precision,
    "test_f1": test_f1,
    "true_neg": int(cm[0, 0]),
    "false_pos": int(cm[0, 1]),
    "false_neg": int(cm[1, 0]),
    "true_pos": int(cm[1, 1]),
    "test_recall_at_050": recall_score(y_test, y_test_pred_05, zero_division=0),
    "test_precision_at_050": precision_score(y_test, y_test_pred_05, zero_division=0),
})

# ── Coefficient table ─────────────────────────────────────────────────────────
coef_table = wandb.Table(
    columns=["feature", "coefficient"],
    data=[[f, float(c)] for f, c in zip(selected_features, lr_model.coef_[0])],
)
wandb.log({"lr_coefficients": coef_table})

# ── Threshold analysis table ──────────────────────────────────────────────────
threshold_data = []
for t in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    y_pred_t = (y_val_proba >= t).astype(int)
    threshold_data.append([
        t,
        recall_score(y_val, y_pred_t, zero_division=0),
        precision_score(y_val, y_pred_t, zero_division=0),
        f1_score(y_val, y_pred_t, zero_division=0),
        y_pred_t.mean(),
    ])

wandb.log({
    "threshold_analysis": wandb.Table(
        columns=["Threshold", "Recall", "Precision", "F1", "Pct_Flagged"],
        data=threshold_data,
    )
})

# ── Interactive PR curve and confusion matrix ─────────────────────────────────
y_probas_both = lr_model.predict_proba(X_test_final)
wandb.log({
    "pr_curve": wandb.plot.pr_curve(
        y_true=y_test.values,
        y_probas=y_probas_both,
        labels=["No Diabetes", "Diabetes"],
    )
})
wandb.log({
    "confusion_matrix": wandb.plot.confusion_matrix(
        y_true=y_test.values,
        preds=y_test_pred,
        class_names=["No Diabetes", "Diabetes"],
    )
})

# ── Save charts to disk and log as artifact ───────────────────────────────────
chart_dir = "wandb_charts"
os.makedirs(chart_dir, exist_ok=True)

charts = {
    "lr_25features_pr_threshold": fig_pr,
    "lr_25features_confusion_matrix": fig_cm,
    "lr_25features_coefficients": fig_coef,
}

for name, figure in charts.items():
    figure.savefig(f"{chart_dir}/{name}.png", dpi=150, bbox_inches="tight")

artifact = wandb.Artifact(
    name=f"{STUDY_NAME}-charts",
    type="evaluation-charts",
    description="Evaluation charts for LR 25-feature model (VIF + backward BIC)",
    metadata={
        "threshold_optimal": float(optimal_thresh),
        "test_recall": float(test_recall),
        "test_precision": float(test_precision),
        "test_ap": float(test_ap),
        "vif_dropped": [f[0] for f in vif_dropped],
        "n_features_final": len(selected_features),
    },
)
artifact.add_dir(chart_dir)
wandb.log_artifact(artifact)

wandb.log({name: wandb.Image(figure) for name, figure in charts.items()})

wandb.summary.update({
    "test_recall": float(test_recall),
    "test_precision": float(test_precision),
    "test_f1": float(test_f1),
    "test_ap": float(test_ap),
    "val_ap": float(val_ap),
    "chosen_threshold": float(optimal_thresh),
    "n_features_initial": X_train.shape[1],
    "n_features_post_vif": len(vif_selected_cols),
    "n_features_final": len(selected_features),
    "model_type": "logistic_regression",
})

wandb.finish()
print("✅ W&B logging complete")